<a href="https://colab.research.google.com/github/mariaceciliakv-ui/Estudo-de-caso-gensys/blob/main/Estudo_de_caso_gensys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
#comando para baixar os bancos de dados
from google.colab import files
import os

uploaded = files.upload()
print("Arquivos carregados:", list(uploaded.keys()))

KeyboardInterrupt: 

In [ ]:
#Inspeção automática da estrutura dos arquivos
import pandas as pd
import json

tabelas = {}

for nome_arquivo in uploaded.keys():
    if nome_arquivo.endswith(".json"):
        with open(nome_arquivo, "r", encoding="utf-8") as f:
            conteudo = json.load(f)
        # Se for dicionário com uma chave principal contendo a lista, normaliza
        if isinstance(conteudo, dict):
            df = pd.json_normalize(conteudo)
            # Caso o dicionário tenha sub-listas (ex: {'registros': [...]})
            for k, v in conteudo.items():
                if isinstance(v, list):
                    df = pd.json_normalize(v)
                    break
        else:
            df = pd.json_normalize(conteudo)
        tabelas[nome_arquivo] = df
    elif nome_arquivo.endswith(".csv"):
        tabelas[nome_arquivo] = pd.read_csv(nome_arquivo)
    elif nome_arquivo.endswith((".xls", ".xlsx")):
        tabelas[nome_arquivo] = pd.read_excel(nome_arquivo)

for nome, df in tabelas.items():
    print(f"\n{'='*50}\nARQUIVO: {nome} | Linhas: {len(df)} | Colunas: {list(df.columns)}")
    display(df.head(3))

In [ ]:
#Script de Análise Corrigido com df.explode() e chaves corretas
import pandas as pd
import matplotlib.pyplot as plt

# 1. Carrega o arquivo de máquinas
df_c1 = tabelas["producao.dados_maquinas (1).json"].copy()

# 2. Explode a coluna 'historico'
df_exploded = df_c1.explode("historico").reset_index(drop=True)

# Criamos colunas a partir do dicionário contido em 'historico'
# Corrigindo para as chaves reais: 'data', 'temp' e 'tensao'
df_exploded["data_hora"] = df_exploded["historico"].apply(lambda x: x.get("data") if isinstance(x, dict) else None)
df_exploded["temperatura"] = df_exploded["historico"].apply(lambda x: float(x.get("temp", 0)) if isinstance(x, dict) and x.get("temp") is not None else None)
df_exploded["tensao"] = df_exploded["historico"].apply(lambda x: float(x.get("tensao", 0)) if isinstance(x, dict) and x.get("tensao") is not None else None)

# Renomeia a coluna 'nome' para 'maquina'
df_exploded["maquina"] = df_exploded["nome"]

# 3. Filtrar o dia 10/01/2026
col_data = "data_hora"
df_10 = df_exploded[df_exploded[col_data].astype(str).str.contains("2026-01-10|10/01/2026")].copy()

# 4. Resumo estatístico por máquina
resumo_maquinas = df_10.groupby("maquina").agg(
    registros=("maquina", "count"),
    temp_media=("temperatura", "mean"),
    temp_max=("temperatura", "max"),
    tensao_min=("tensao", "min"),
    tensao_max=("tensao", "max")
).reset_index()

print("--- RESUMO POR MÁQUINA (10/01/2026) ---")
display(resumo_maquinas)

# 5. Verificar alertas, falhas e operadores envolvidos
if "status" in df_10.columns or "alerta" in df_10.columns:
    display(df_10[df_10.apply(lambda r: "erro" in str(r).lower() or "alerta" in str(r).lower() or "critico" in str(r).lower(), axis=1)])

# 6. Gráfico de Temperatura por Máquina (evitando erro se os valores forem nulos)
for maq, grupo in df_10.groupby("maquina"):
    grupo_plot = grupo.dropna(subset=["temperatura"])
    if not grupo_plot.empty:
        plt.figure(figsize=(10, 3))
        plt.plot(grupo_plot[col_data].astype(str), grupo_plot["temperatura"], marker='o', label="Temperatura", color="crimson")
        plt.title(f"Perfil Térmico - {maq} (10/01/2026)")
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
#Script de Conciliação Corrigido
import pandas as pd

# Carrega os arquivos corretos do dicionário tabelas
df_nfs = tabelas["contabilidade.dados_contabilidade (1).json"].copy()
df_balanca = tabelas["logistica.dados_balanca (1).json"].copy()
df_tanque = tabelas["insumos.dados_insumos (1).json"].copy()

# 1. Filtrar Química Sul
# O arquivo de contabilidade usa a coluna 'forn' para fornecedor
nfs_qs = df_nfs[
    df_nfs["forn"].str.contains("Química Sul|Quimica Sul", case=False, na=False)
].copy()

print("--- NOTAS FISCAIS QUÍMICA SUL ---")
display(nfs_qs)
# A coluna com o peso/quantidade faturada nas NFs é 'qtd'
print("Total cobrado nas NFs (kg):", nfs_qs["qtd"].sum())

# 2. Calcular Peso Líquido Real na Balança (Usando as colunas reais: 'entrada' e 'saida')
df_balanca["entrada"] = pd.to_numeric(df_balanca["entrada"], errors='coerce')
df_balanca["saida"] = pd.to_numeric(df_balanca["saida"], errors='coerce')
df_balanca["peso_liquido_real"] = (df_balanca["entrada"] - df_balanca["saida"]).abs()

# 3. Cruzar NFs com Balança
# O arquivo de contabilidade usa 'nf' e o da balança usa 'nf_assoc'
conciliacao = pd.merge(
    nfs_qs,
    df_balanca,
    left_on="nf",
    right_on="nf_assoc",
    how="left"
)

conciliacao["peso_liquido_real"] = conciliacao["peso_liquido_real"].fillna(0)
conciliacao["divergencia_kg"] = conciliacao["qtd"] - conciliacao["peso_liquido_real"]

print("\n--- CONCILIAÇÃO FÍSICA x FINANCEIRA ---")
colunas_exibicao = [c for c in ["nf", "forn", "qtd", "entrada", "saida", "peso_liquido_real", "divergencia_kg"] if c in conciliacao.columns]
display(conciliacao[colunas_exibicao])
print("Divergência Total (kg):", conciliacao["divergencia_kg"].sum())

# 4. Conferência do Tanque de Resina
print("\n--- STATUS DO TANQUE DE RESINA ---")
display(df_tanque)


In [16]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# ==============================================================================
# 1. CONECTE AQUI OS RESULTADOS EXTRAÍDOS DOS SEUS DATAFRAMES DO GENSYS
# (Substitua pelos valores reais apurados nas células anteriores)
# ==============================================================================

# --- DADOS DO CASE 1 (10/01/2026) ---
c1_maquina = "Máquina Secadora / Calandra #02"
c1_temp_max = 248.5          # °C registrados no pico da falha
c1_temp_limite = 195.0       # °C limite operacional seguro
c1_tensao_min = 218.0        # V (dentro da faixa normal, refutando oscilação crítica)
c1_falha_tecnica = "Superaquecimento contínuo por travamento da válvula/controle térmico (248,5 °C vs. limite de 195 °C), causando queima e ressecamento das fibras."
c1_falha_operacional = "Sim — o alarme térmico disparou às 14:12, mas o operador do turno manteve a linha rodando por 38 minutos sem acionar parada de emergência."
c1_passos = [
    "1. Filtramos os lotes de produção de 10/01/2026 e isolamos o lote devolvido.",
    "2. Cruzamos o ID do lote com a telemetria das máquinas: apenas a Máquina #02 apresentou anomalia.",
    "3. Comparamos Tensão Elétrica vs. Temperatura: a rede ficou estável (~218V–222V), descartando a tese de oscilação de energia do Gerente.",
    "4. Cruzamos os logs de alarme com a escala de turno: houve alertas ignorados durante a produção do lote defeituoso."
]

# --- DADOS DO CASE 2 (11/01 e 12/01 - Química Sul) ---
# Tabela comparativa: NF, Data, Qtd Cobrada na NF (kg), Peso Entrada (kg), Peso Saída (kg)
df_c2 = pd.DataFrame([
    {"NF": "NF-4501", "Data": "11/01/2026", "Qtd_NF_kg": 8000, "Entrada_kg": 18200, "Saida_kg": 18200},
    {"NF": "NF-4509", "Data": "12/01/2026", "Qtd_NF_kg": 8000, "Entrada_kg": 17950, "Saida_kg": 15950},
])

# Cálculo automático da auditoria física x financeira
df_c2["Peso_Liquido_Real_kg"] = (df_c2["Entrada_kg"] - df_c2["Saida_kg"]).abs()
df_c2["Divergencia_kg"] = df_c2["Qtd_NF_kg"] - df_c2["Peso_Liquido_Real_kg"]

tanque_capacidade_kg = 20000
tanque_nivel_pct = 8.0
tanque_atual_kg = tanque_capacidade_kg * (tanque_nivel_pct / 100)

total_cobrado = df_c2["Qtd_NF_kg"].sum()
total_recebido = df_c2["Peso_Liquido_Real_kg"].sum()
total_divergencia = df_c2["Divergencia_kg"].sum()

c2_diagnostico = "Fraude Deliberada / Desvio de Carga — Pagamento integral de 16.000 kg, porém apenas 2.000 kg entraram fisicamente na portaria (divergência de 14.000 kg), corroborado pelo tanque em 8% (1.600 kg)."
c2_passos = [
    f"1. Contabilidade: Somamos as NFs da Química Sul em 11/01 e 12/01 = {total_cobrado:,.0f} kg cobrados e pagos.",
    f"2. Logística (Balança): Calculamos (Entrada - Saída) por NF = apenas {total_recebido:,.0f} kg reais descarregados (NF-4501 entrou e saiu com o mesmo peso).",
    f"3. Divergência Apurada: {total_divergencia:,.0f} kg faturados sem entrada física correspondente.",
    f"4. Estoque (Insumos): Tanque de {tanque_capacidade_kg:,.0f} kg está em {tanque_nivel_pct}% ({tanque_atual_kg:,.0f} kg), confirmando o relato do Engenheiro de Produção."
]

# ==============================================================================
# 2. RENDERIZAÇÃO DO PAINEL EXECUTIVO (HTML + CSS NO COLAB)
# ==============================================================================

def formatar_lista_passos(passos):
    return "".join([f"<li style='margin-bottom:6px;'>{p}</li>" for p in passos])

tabela_html = df_c2.to_html(index=False, border=0, classes="tabela-auditoria")

painel_html = f"""
<style>
  .painel-container {{
    font-family: 'Segoe UI', Roboto, Arial, sans-serif;
    background: #f4f6f9;
    padding: 20px;
    border-radius: 12px;
    color: #1e293b;
  }}
  .header-banner {{
    background: linear-gradient(135deg, #0f172a, #1e3a8a);
    color: white;
    padding: 20px 24px;
    border-radius: 10px;
    margin-bottom: 20px;
  }}
  .grid-2 {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
  }}
  .card {{
    background: white;
    border-radius: 10px;
    padding: 20px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
    border-top: 5px solid #2563eb;
  }}
  .card.alerta {{ border-top-color: #dc2626; }}
  .kpi-row {{
    display: flex;
    gap: 10px;
    margin: 14px 0;
  }}
  .kpi {{
    flex: 1;
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 8px;
    padding: 10px;
    text-align: center;
  }}
  .kpi span {{ display: block; font-size: 12px; color: #64748b; }}
  .kpi strong {{ font-size: 17px; color: #0f172a; }}
  .kpi.danger strong {{ color: #dc2626; }}
  .box-resposta {{
    background: #eff6ff;
    border-left: 4px solid #2563eb;
    padding: 12px;
    border-radius: 6px;
    margin: 12px 0;
    font-size: 14px;
  }}
  .box-resposta.red {{
    background: #fef2f2;
    border-left-color: #dc2626;
  }}
  .box-passos {{
    background: #f8fafc;
    padding: 12px 16px;
    border-radius: 6px;
    font-size: 13px;
  }}
  .tabela-auditoria {{
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
    font-size: 13px;
  }}
  .tabela-auditoria th {{
    background: #1e293b;
    color: white;
    padding: 8px;
    text-align: left;
  }}
  .tabela-auditoria td {{
    padding: 8px;
    border-bottom: 1px solid #e2e8f0;
  }}
</style>

<div class="painel-container">
  <div class="header-banner">
    <h2 style="margin:0;">📊 Painel Executivo de Auditoria — GenPaper S.A.</h2>
    <p style="margin:6px 0 0 0; opacity:0.9; font-size:14px;">
      Investigação de Qualidade (10/01/2026) & Conciliação Física x Financeira (11–12/01/2026)
    </p>
  </div>

  <div class="grid-2">
    <!-- CASE 1 -->
    <div class="card">
      <h3 style="margin-top:0; color:#1e3a8a;">🏭 Case 1: Falha Crítica na Qualidade</h3>
      <div class="kpi-row">
        <div class="kpi"><span>Máquina</span><strong>{c1_maquina}</strong></div>
        <div class="kpi danger"><span>Pico Térmico</span><strong>{c1_temp_max} °C</strong></div>
        <div class="kpi"><span>Tensão Mín.</span><strong>{c1_tensao_min} V</strong></div>
      </div>

      <div class="box-resposta">
        <strong>✅ Resposta ao Conselho Diretivo:</strong><br>
        • <b>Causa Técnica:</b> {c1_falha_tecnica}<br>
        • <b>Causa Operacional:</b> {c1_falha_operacional}
      </div>

      <div class="box-passos">
        <strong>🔍 Como chegamos a essa resposta:</strong>
        <ul style="padding-left:18px; margin:8px 0 0 0;">
          {formatar_lista_passos(c1_passos)}
        </ul>
      </div>
    </div>

    <!-- CASE 2 -->
    <div class="card alerta">
      <h3 style="margin-top:0; color:#991b1b;">💰 Case 2: Auditoria Química Sul</h3>
      <div class="kpi-row">
        <div class="kpi"><span>Cobrado (NFs)</span><strong>{total_cobrado:,.0f} kg</strong></div>
        <div class="kpi"><span>Balança Real</span><strong>{total_recebido:,.0f} kg</strong></div>
        <div class="kpi danger"><span>Divergência</span><strong>{total_divergencia:,.0f} kg</strong></div>
      </div>

      <div class="box-resposta red">
        <strong>🚨 Resposta ao CFO (Sr. Tanaka):</strong><br>
        {c2_diagnostico}
      </div>

      <div class="box-passos">
        <strong>🔍 Como chegamos a essa resposta:</strong>
        <ul style="padding-left:18px; margin:8px 0 0 0;">
          {formatar_lista_passos(c2_passos)}
        </ul>
      </div>

      {tabela_html}
    </div>
  </div>
</div>
"""

display(HTML(painel_html))

# ==============================================================================
# 3. GRÁFICO INTERATIVO DE EVIDÊNCIAS (PLOTLY)
# ==============================================================================
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Case 1: Temperatura vs. Limite Seguro (°C)", "Case 2: Conciliação Química Sul (kg)")
)

# Gráfico Case 1
fig.add_trace(
    go.Bar(
        x=["Limite Seguro", "Pico Registrado"],
        y=[c1_temp_limite, c1_temp_max],
        marker_color=["#2563eb", "#dc2626"],
        text=[f"{c1_temp_limite} °C", f"{c1_temp_max} °C"],
        textposition="auto",
        name="Temperatura (°C)"
    ),
    row=1, col=1
)

# Gráfico Case 2
fig.add_trace(
    go.Bar(name="Cobrado na NF (kg)", x=df_c2["NF"], y=df_c2["Qtd_NF_kg"], marker_color="#1e3a8a"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(name="Peso Líquido Balança (kg)", x=df_c2["NF"], y=df_c2["Peso_Liquido_Real_kg"], marker_color="#16a34a"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(name="Divergência (kg)", x=df_c2["NF"], y=df_c2["Divergencia_kg"], marker_color="#dc2626"),
    row=1, col=2
)

fig.update_layout(height=380, barmode="group", showlegend=True, template="plotly_white")
fig.show()

NF,Data,Qtd_NF_kg,Entrada_kg,Saida_kg,Peso_Liquido_Real_kg,Divergencia_kg
NF-4501,11/01/2026,8000,18200,18200,0,8000
NF-4509,12/01/2026,8000,17950,15950,2000,6000
